In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Chargement des données RFM
df = pd.read_csv('../data/data_clean.csv', parse_dates=['InvoiceDate'])

from datetime import timedelta
reference_date = df['InvoiceDate'].max() + timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recence = ('InvoiceDate', lambda x: (reference_date-x.max()).days),
    Frequence = ('InvoiceNo', 'nunique'),
    Montant = ('TotalPrice', 'sum')
).reset_index()


rfm['R_score'] = pd.qcut(rfm['Recence'], q=4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequence'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Montant'], q=4, labels=[1,2,3,4])

def segmenter_client(row):
    r = int(row['R_score'])
    f = int(row['F_score'])
    m = int(row['M_score'])
    if r >= 3 and f >=3 and m >=3:
        return 'Champions'
    